# S4 · AndinaLog 03B · Notebook 1 · Diagnóstico de conductores

Este notebook trabaja **solo** con `andinalog_hr_drivers.csv`. Lee la capa Bronze desde `datasets/AndinaLog_03B_Bronce/`, detecta problemas y conserva intactos los seis campos originales. No normaliza identificadores, no corrige nombres, no elimina duplicados y no modifica valores numéricos. El tratamiento corresponde al notebook 2 después de aprobar sus reglas.

Cada ejecución reemplaza cuatro archivos en `S4/andinalog_hr_drivers/notebook1/salidas/`:

1. `andinalog_hr_drivers_diagnosticado.csv`: todas las filas, los campos originales y solo `fila_bronze`, `columnas_con_problemas` y `en_cuarentena`.
2. `andinalog_hr_drivers_problemas.csv`: una fila por problema, con columna, código estable y evidencia.
3. `andinalog_hr_drivers_cuarentena.csv`: extracto informativo de las filas marcadas.
4. `andinalog_hr_drivers_reporte_calidad.csv`: conteos y huella SHA-256 del CSV de origen.


## 1 · Configuración y origen

`ENTORNO = "auto"` usa Google Drive cuando se ejecuta en Colab y busca la raíz del repositorio cuando se ejecuta localmente. En Colab, ajusta `RUTA_PROYECTO_DRIVE` a la carpeta que contiene `datasets/` y `S4/`.


In [2]:
from pathlib import Path
import hashlib
import os
import tempfile
import pandas as pd

ENTORNO = "auto"  # "auto", "local" o "drive"
RUTA_PROYECTO_DRIVE = "/content/drive/MyDrive/GIAD"
CARPETA_DATASETS = "AndinaLog_03B_Bronce"
NOMBRE_CSV = "andinalog_hr_drivers.csv"
VERSION_DIAGNOSTICO = "GIAD-M3-S4-HR-drivers-diagnostico-v1"

COLUMNAS_ORIGINALES = [
    "chofer_id", "chofer_nombre", "centro_distribucion",
    "horas_conduccion_mes", "salario_base_bob", "ausentismo_dias",
]

def encontrar_raiz_local():
    for carpeta in [Path.cwd(), *Path.cwd().parents]:
        if (carpeta / "datasets" / CARPETA_DATASETS).is_dir() and (carpeta / "S4").is_dir():
            return carpeta
    raise FileNotFoundError("No se encontró la raíz del proyecto; ejecuta dentro de practicasNotebookColab.")

def configurar_rutas(entorno, ruta_drive):
    if entorno == "auto":
        entorno = "drive" if "google.colab" in __import__("sys").modules else "local"
    if entorno == "drive":
        from google.colab import drive
        drive.mount("/content/drive")
        raiz = Path(ruta_drive)
    elif entorno == "local":
        raiz = encontrar_raiz_local()
    else:
        raise ValueError("ENTORNO debe ser 'auto', 'local' o 'drive'")
    bronze = raiz / "datasets" / CARPETA_DATASETS / NOMBRE_CSV
    salidas = raiz / "S4" / "andinalog_hr_drivers" / "notebook1" / "salidas"
    if not bronze.is_file():
        raise FileNotFoundError(f"No se encontró el CSV Bronze: {bronze}")
    return bronze, salidas

RUTA_BRONZE, DIRECTORIO_SALIDAS = configurar_rutas(ENTORNO, RUTA_PROYECTO_DRIVE)
print("Bronze:", RUTA_BRONZE)
print("Salidas:", DIRECTORIO_SALIDAS)


Bronze: c:\Users\remrodri\Github\practicasNotebookColab\datasets\AndinaLog_03B_Bronce\andinalog_hr_drivers.csv
Salidas: c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_hr_drivers\notebook1\salidas


## 2 · Carga y contrato

La lectura mantiene todas las columnas como texto y los vacíos como cadenas vacías. Las conversiones numéricas usadas para diagnosticar son temporales y no sustituyen los valores Bronze.


In [3]:
def cargar_bronze(ruta):
    huella = hashlib.sha256(ruta.read_bytes()).hexdigest()
    df = pd.read_csv(ruta, dtype="string", encoding="utf-8-sig", keep_default_na=False)
    return df, huella

def validar_esquema(df):
    if list(df.columns) != COLUMNAS_ORIGINALES:
        faltantes = sorted(set(COLUMNAS_ORIGINALES) - set(df.columns))
        extras = sorted(set(df.columns) - set(COLUMNAS_ORIGINALES))
        raise ValueError(f"Esquema inesperado. Faltantes: {faltantes}; extras: {extras}; orden: {list(df.columns)}")
    if not df.columns.is_unique:
        raise ValueError("Hay nombres de columnas duplicados")
    return df

df_bronze, HASH_BRONZE = cargar_bronze(RUTA_BRONZE)
validar_esquema(df_bronze)
print(f"Bronze: {len(df_bronze):,} filas × {len(df_bronze.columns)} columnas")
print("SHA-256:", HASH_BRONZE)
display(df_bronze.head())


Bronze: 156 filas × 6 columnas
SHA-256: f4edf8a0f50292e3f320629f26b30a1a725df7e430965465faf044d877c1a718


,chofer_id,chofer_nombre,centro_distribucion,horas_conduccion_mes,salario_base_bob,ausentismo_dias
0,CHO-001,Oscar Calle,La Paz,120.6,5901.26,0
1,CHO-002,Hugo Banzer,La Paz,136.4,3931.46,1
2,CHO-003,Marcos Lino,Oruro,121.4,6269.76,1
3,CHO-004,Adolfo Perez,Santa Cruz,195.5,4940.3,3
4,CHO-005,Rene Rojas,La Paz,164.8,6144.78,0


## 3 · Catálogo y reglas de diagnóstico

Los códigos son estables para que el notebook 2 pueda reconocer cada problema. Las reglas detectan estructura, duplicidad y dominios básicos; no establecen un máximo de horas mensuales porque ese límite requiere una regla de negocio aprobada.


In [4]:
CATALOGO_PROBLEMAS = pd.DataFrame([
    ("chofer_id", "FALTANTE", "Identificador vacío"),
    ("chofer_id", "FORMATO_INVALIDO", "No cumple CHO-### exactamente"),
    ("chofer_id", "DUPLICADO", "Identificador repetido; se marca la aparición posterior"),
    ("chofer_nombre", "FALTANTE", "Nombre vacío"),
    ("chofer_nombre", "FORMATO_INVALIDO", "No contiene al menos dos palabras formadas por letras"),
    ("centro_distribucion", "FALTANTE", "Centro vacío"),
    ("centro_distribucion", "VALOR_NO_RECONOCIDO", "Centro fuera del dominio operativo declarado"),
    ("horas_conduccion_mes", "FALTANTE", "Valor vacío"),
    ("horas_conduccion_mes", "NO_NUMERICO", "Valor no convertible a número"),
    ("horas_conduccion_mes", "NEGATIVO", "Cantidad de horas menor que cero"),
    ("salario_base_bob", "FALTANTE", "Valor vacío"),
    ("salario_base_bob", "NO_NUMERICO", "Valor no convertible a número"),
    ("salario_base_bob", "NO_POSITIVO", "Salario menor o igual que cero"),
    ("ausentismo_dias", "FALTANTE", "Valor vacío"),
    ("ausentismo_dias", "NO_NUMERICO", "Valor no convertible a número"),
    ("ausentismo_dias", "NEGATIVO", "Cantidad de días menor que cero"),
    ("ausentismo_dias", "NO_ENTERO", "La cantidad de días contiene una fracción"),
], columns=["columna_afectada", "codigo_error", "criterio"])
display(CATALOGO_PROBLEMAS)

def texto(df, columna):
    return df[columna].astype("string").str.strip()

def registrar_problema(df, mascara, columna, codigo, evidencia=None):
    mascara = mascara.fillna(False).astype(bool)
    filas = df.loc[mascara, ["fila_bronze"]].copy()
    filas["columna_afectada"] = columna
    filas["codigo_error"] = codigo
    if evidencia is None:
        evidencia = df[columna] if columna in df.columns else pd.Series("", index=df.index, dtype="string")
    filas["valor_original"] = evidencia.loc[mascara].astype("string").to_numpy()
    return filas

def detectar_identificador_y_duplicados(df):
    original = df["chofer_id"].astype("string")
    limpio = original.str.strip()
    return [
        registrar_problema(df, limpio.eq(""), "chofer_id", "FALTANTE", original),
        registrar_problema(df, limpio.ne("") & ~original.str.fullmatch(r"CHO-\d{3}").fillna(False), "chofer_id", "FORMATO_INVALIDO", original),
        registrar_problema(df, limpio.duplicated(keep="first"), "chofer_id", "DUPLICADO", original),
    ]

def detectar_nombre(df):
    original = df["chofer_nombre"].astype("string")
    limpio = original.str.strip()
    patron = r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+(?: [A-Za-zÁÉÍÓÚÜÑáéíóúüñ]+)+"
    return [
        registrar_problema(df, limpio.eq(""), "chofer_nombre", "FALTANTE", original),
        registrar_problema(df, limpio.ne("") & ~original.str.fullmatch(patron).fillna(False), "chofer_nombre", "FORMATO_INVALIDO", original),
    ]

def detectar_centro(df):
    original = df["centro_distribucion"].astype("string")
    limpio = original.str.strip()
    permitidos = {"La Paz", "Cochabamba", "Santa Cruz", "Oruro", "Tarija"}
    return [
        registrar_problema(df, limpio.eq(""), "centro_distribucion", "FALTANTE", original),
        registrar_problema(df, limpio.ne("") & ~original.isin(permitidos), "centro_distribucion", "VALOR_NO_RECONOCIDO", original),
    ]

def detectar_numero(df, columna, minimo, permite_cero, exige_entero=False):
    original = df[columna].astype("string")
    limpio = original.str.strip()
    numero = pd.to_numeric(limpio, errors="coerce")
    codigo_limite = "NEGATIVO" if permite_cero else "NO_POSITIVO"
    fuera = numero.lt(minimo) if permite_cero else numero.le(minimo)
    hallazgos = [
        registrar_problema(df, limpio.eq(""), columna, "FALTANTE", original),
        registrar_problema(df, limpio.ne("") & numero.isna(), columna, "NO_NUMERICO", original),
        registrar_problema(df, numero.notna() & fuera, columna, codigo_limite, original),
    ]
    if exige_entero:
        hallazgos.append(registrar_problema(df, numero.notna() & numero.mod(1).ne(0), columna, "NO_ENTERO", original))
    return hallazgos

def diagnosticar(df_bronze):
    principal = df_bronze.copy(deep=True)
    principal.insert(0, "fila_bronze", range(1, len(principal) + 1))
    hallazgos = (
        detectar_identificador_y_duplicados(principal)
        + detectar_nombre(principal)
        + detectar_centro(principal)
        + detectar_numero(principal, "horas_conduccion_mes", 0, permite_cero=True)
        + detectar_numero(principal, "salario_base_bob", 0, permite_cero=False)
        + detectar_numero(principal, "ausentismo_dias", 0, permite_cero=True, exige_entero=True)
    )
    problemas = pd.concat(hallazgos, ignore_index=True)
    problemas = problemas.sort_values(["fila_bronze", "columna_afectada", "codigo_error"], kind="stable").reset_index(drop=True)
    problemas["version_diagnostico"] = VERSION_DIAGNOSTICO
    columnas_por_fila = problemas.groupby("fila_bronze")["columna_afectada"].agg(lambda valores: "|".join(dict.fromkeys(valores)))
    principal["columnas_con_problemas"] = principal["fila_bronze"].map(columnas_por_fila).fillna("")
    principal["en_cuarentena"] = principal["columnas_con_problemas"].ne("")
    return principal, problemas

df_diagnosticado, df_problemas = diagnosticar(df_bronze)
df_cuarentena = df_diagnosticado.loc[df_diagnosticado["en_cuarentena"]].copy()
print(f"Principal: {len(df_diagnosticado):,}; problemas: {len(df_problemas):,}; filas en cuarentena: {len(df_cuarentena):,}")
display(df_problemas.groupby(["columna_afectada", "codigo_error"]).size().rename("filas").reset_index())


,columna_afectada,codigo_error,criterio
0,chofer_id,FALTANTE,Identificador vacío
1,chofer_id,FORMATO_INVALIDO,No cumple CHO-### exactamente
2,chofer_id,DUPLICADO,Identificador repetido; se marca la aparición ...
3,chofer_nombre,FALTANTE,Nombre vacío
4,chofer_nombre,FORMATO_INVALIDO,No contiene al menos dos palabras formadas por...
5,centro_distribucion,FALTANTE,Centro vacío
6,centro_distribucion,VALOR_NO_RECONOCIDO,Centro fuera del dominio operativo declarado
7,horas_conduccion_mes,FALTANTE,Valor vacío
8,horas_conduccion_mes,NO_NUMERICO,Valor no convertible a número
9,horas_conduccion_mes,NEGATIVO,Cantidad de horas menor que cero


Principal: 156; problemas: 10; filas en cuarentena: 10


,columna_afectada,codigo_error,filas
0,chofer_id,DUPLICADO,5
1,chofer_id,FORMATO_INVALIDO,5


## 4 · Reporte y comprobaciones antes de exportar

La huella SHA-256 identifica la versión exacta del CSV Bronze. Una fila puede acumular varios hallazgos.


In [5]:
def construir_reporte(df_bronze, principal, problemas, ruta, huella):
    conteos = problemas.groupby(["columna_afectada", "codigo_error"]).size()
    datos = [
        ("archivo_bronze", ruta.name), ("sha256_bronze", huella),
        ("version_diagnostico", VERSION_DIAGNOSTICO), ("filas_bronze", len(df_bronze)),
        ("filas_diagnosticadas", len(principal)),
        ("filas_en_cuarentena", int(principal["en_cuarentena"].sum())),
        ("filas_sin_cuarentena", int((~principal["en_cuarentena"]).sum())),
        ("problemas_detectados", len(problemas)),
    ]
    datos += [(f"{col}:{codigo}", int(total)) for (col, codigo), total in conteos.items()]
    return pd.DataFrame(datos, columns=["metrica", "valor"])

def validar_resultados(df_bronze, principal, problemas, cuarentena, reporte):
    assert list(principal.columns) == ["fila_bronze", *COLUMNAS_ORIGINALES, "columnas_con_problemas", "en_cuarentena"]
    pd.testing.assert_frame_equal(principal[COLUMNAS_ORIGINALES], df_bronze[COLUMNAS_ORIGINALES])
    assert len(principal) == len(df_bronze)
    assert principal["fila_bronze"].is_unique
    assert len(cuarentena) == int(principal["en_cuarentena"].sum())
    assert problemas["fila_bronze"].isin(principal["fila_bronze"]).all()
    assert problemas[["columna_afectada", "codigo_error"]].apply(tuple, axis=1).isin(CATALOGO_PROBLEMAS[["columna_afectada", "codigo_error"]].apply(tuple, axis=1)).all()
    assert set(problemas["fila_bronze"]) == set(cuarentena["fila_bronze"])
    assert len(reporte) >= 8

reporte_calidad = construir_reporte(df_bronze, df_diagnosticado, df_problemas, RUTA_BRONZE, HASH_BRONZE)
validar_resultados(df_bronze, df_diagnosticado, df_problemas, df_cuarentena, reporte_calidad)
display(reporte_calidad)
print("Comprobaciones previas a la exportación: correctas")


,metrica,valor
0,archivo_bronze,andinalog_hr_drivers.csv
1,sha256_bronze,f4edf8a0f50292e3f320629f26b30a1a725df7e4309654...
2,version_diagnostico,GIAD-M3-S4-HR-drivers-diagnostico-v1
3,filas_bronze,156
4,filas_diagnosticadas,156
5,filas_en_cuarentena,10
6,filas_sin_cuarentena,146
7,problemas_detectados,10
8,chofer_id:DUPLICADO,5
9,chofer_id:FORMATO_INVALIDO,5


Comprobaciones previas a la exportación: correctas


## 5 · Exportación reproducible

Los cuatro CSV se escriben primero como archivos temporales y luego reemplazan las salidas anteriores con los mismos nombres. El CSV Bronze nunca se sobrescribe.


In [6]:
def exportar_salidas(directorio, tablas, ruta_bronze, huella_inicial):
    if hashlib.sha256(ruta_bronze.read_bytes()).hexdigest() != huella_inicial:
        raise RuntimeError("El CSV Bronze cambió durante la ejecución; no se exportarán resultados")
    directorio.mkdir(parents=True, exist_ok=True)
    temporales = {}
    try:
        for nombre, tabla in tablas.items():
            destino = directorio / nombre
            with tempfile.NamedTemporaryFile(mode="w", suffix=".csv", prefix=".tmp_hr_", dir=directorio, encoding="utf-8-sig", newline="", delete=False) as tmp:
                tabla.to_csv(tmp, index=False)
                temporales[destino] = Path(tmp.name)
        for destino, temporal in temporales.items():
            os.replace(temporal, destino)
    finally:
        for temporal in temporales.values():
            temporal.unlink(missing_ok=True)
    return list(temporales)

tablas_salida = {
    "andinalog_hr_drivers_diagnosticado.csv": df_diagnosticado,
    "andinalog_hr_drivers_problemas.csv": df_problemas,
    "andinalog_hr_drivers_cuarentena.csv": df_cuarentena,
    "andinalog_hr_drivers_reporte_calidad.csv": reporte_calidad,
}
rutas_creadas = exportar_salidas(DIRECTORIO_SALIDAS, tablas_salida, RUTA_BRONZE, HASH_BRONZE)
for ruta in rutas_creadas:
    print(ruta)
print("Bronze intacta; salidas anteriores reemplazadas")


c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_hr_drivers\notebook1\salidas\andinalog_hr_drivers_diagnosticado.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_hr_drivers\notebook1\salidas\andinalog_hr_drivers_problemas.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_hr_drivers\notebook1\salidas\andinalog_hr_drivers_cuarentena.csv
c:\Users\remrodri\Github\practicasNotebookColab\S4\andinalog_hr_drivers\notebook1\salidas\andinalog_hr_drivers_reporte_calidad.csv
Bronze intacta; salidas anteriores reemplazadas


## Siguiente etapa

El notebook 2 leerá el archivo diagnosticado y el detalle de problemas. Ninguna corrección queda aprobada automáticamente. Una fila solo saldrá de cuarentena cuando todos sus problemas hayan sido resueltos y validados con reglas acordadas.
